# Market-derived power ratings (#50)

Thin client over `g_nfl.ml.market_ratings`. Regenerate the parquet with:

```bash
make market-ratings
```

**Sign convention:** higher is better for *both* `off_rating` and `def_rating`
(`def_rating` = points prevented below average). So up-and-to-the-right is good
on the scatter, no axis flipping. This is the opposite of
`ml/features/opponent.py`, whose `def_rating` is EPA allowed.

**Trust `ovr_rating`.** The off/def split comes entirely from `total_line`,
which is thinner than the spread market and conflates pace with efficiency.

In [ ]:
import polars as pl

from g_nfl.ml.market_ratings import DEFAULT_OUTPUT, compare_ratings
from g_nfl.utils.config import CUR_SEASON, CUR_WEEK
from g_nfl.visualisation.plots import plot_scatter

ratings = pl.read_parquet(DEFAULT_OUTPUT)
ratings.tail()

## Where every team sits right now

In [ ]:
season, week = CUR_SEASON, CUR_WEEK

current = ratings.filter(
    (pl.col("season") == season) & (pl.col("week") == week)
).sort("ovr_rating", descending=True)

hfa = current["hfa"][0]
print(f"solved HFA: {hfa:.2f}")
current.select("team", "ovr_rating", "off_rating", "def_rating").with_columns(
    pl.col(pl.Float64).round(2)
)

In [ ]:
# plot_scatter is a pandas consumer — convert at the boundary only
plot_scatter(
    current.to_pandas(),
    x="off_rating",
    y="def_rating",
    title=f"Market-derived power ratings — {season} week {week} (HFA {hfa:.2f})",
    ax_labels=("Offense (pts above avg)", "Defense (pts prevented)"),
    zero_reference=True,
)

## Rating trajectory through the season

How the market moved on a team week to week. Flat stretches are byes (no
games that week, so the ridge holds the team at its prior).

In [ ]:
import matplotlib.pyplot as plt

teams = ["KC", "BUF", "PHI", "DET", "TEN"]

traj = ratings.filter(
    (pl.col("season") == season) & pl.col("team").is_in(teams)
).sort("week")

fig, ax = plt.subplots(figsize=(11, 6))
for team in teams:
    d = traj.filter(pl.col("team") == team)
    ax.plot(d["week"], d["ovr_rating"], marker="o", ms=3, label=team)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set(xlabel="week", ylabel="ovr_rating", title=f"{season} market rating trajectory")
ax.legend()
plt.show()

## Market vs my ratings

Positive `diff` = I am higher on the team than the market is. Swap the
hand-entered frame below for the homers/composite ratings.

In [ ]:
mine = pl.DataFrame(
    {
        "team": ["KC", "BUF", "PHI", "DET", "TEN"],
        "ovr_rating": [6.0, 6.5, 4.0, 8.0, -7.0],
    }
)

compare_ratings(current, mine).with_columns(pl.col(pl.Float64).round(2))